# RAG Application Using  TypeSense


In [20]:
import os
import typesense
from dotenv import load_dotenv

load_dotenv()

client = typesense.Client({
    'nodes': [{
        'host': os.getenv('TYPESENSE_HOST'),
        'port': os.getenv('TYPESENSE_PORT'),
        'protocol': os.getenv('TYPESENSE_PROTOCOL'),
    }],
    'api_key': os.getenv('TYPESENSE_API_KEY'),
    'connection_timeout_seconds': 5,
})

In [4]:
book_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_years', 'type': 'int32', 'facet': True},
        {'name': 'rating_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'},
    ],
    'default_sorting_field': 'rating_count',
}
print(client.collections.create(book_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [5]:
client

In [6]:
with open('books.json', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents

In [8]:
search_parameters = {
    'q': "harry potter",
    'query_by': "title, authors",
    'sort_by': "rating_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 0,
 'hits': [],
 'out_of': 0,
 'page': 1,
 'request_params': {'collection_name': 'books',
  'first_q': 'harry potter',
  'per_page': 10,
  'q': 'harry potter'},
 'search_cutoff': False,
 'search_time_ms': 0}

In [9]:
harry_potter = {
    'id': '16',
    'title': "Harry Potter and the Philosopher's Stone",
    'authors': ['J. K. Rowling'],
    'publication_years': 1997,
    'rating_count': 150000,
    'average_rating': 4.7
}

client.collections['books'].documents.upsert(harry_potter)

{'authors': ['J. K. Rowling'],
 'average_rating': 4.7,
 'id': '16',
 'publication_years': 1997,
 'rating_count': 150000,
 'title': "Harry Potter and the Philosopher's Stone"}

In [24]:
from langchain_community.document_loaders import TextLoader
from langchain_typesense import TypesenseVectorStore
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [25]:
GROQ_API = os.getenv("GROQ_API_KEY")

In [16]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
# docsearch = Typesense.from_documents(
#     docs,
#     embeddings,
#     typesense_client_params={
#         "host": os.getenv("TYPESENSE_HOST"),
#         "port": os.getenv("TYPESENSE_PORT"),
#         "protocol": os.getenv("TYPESENSE_PROTOCOL"),
#         "api_key": os.getenv("TYPESENSE_API_KEY"),
#         "connection_timeout_seconds": 5,
#     },
# )
docsearch = TypesenseVectorStore.from_documents(
    docs,
    embeddings,
    client=client,
    collection_name="rag_documents",
)

In [30]:
query = "files, network connections"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

A rootkit is a type of malicious software designed to hide the presence of an attacker or other malware on a computer while maintaining privileged access, often at the operating-system or kernel level. Rootkits can manipulate system processes, files, network connections, or security tools so that malicious activity is difficult to detect; for example, a kernel-level rootkit on a Linux server could attempt to hide a malicious process from commands such as ps or conceal network activity from normal monitoring tools. Rootkits can exist at different levels, including user-space, kernel-space, boot/firmware, and hypervisor levels, with deeper levels generally having greater control and persistence.


In [31]:
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['TypesenseVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_typesense.vectorstores.TypesenseVectorStore object at 0x7fddfb2e0ad0>, search_kwargs={})

In [ ]:
query = "files, network connections" 
retriever.invo